# Online Retail II Exploration

This notebook loads both yearly sheets from the Online Retail II workbook into pandas, validates and prepares the data, flags questionable rows, and saves reviewable outputs.

## 1. Import Required Libraries

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

## 2. Configure File Paths

In [2]:
PROJECT_ROOT = Path.cwd()
INPUT_PATH = PROJECT_ROOT / "data" / "online_retail_II.xlsx"
OUTPUT_DIR = PROJECT_ROOT / "reports"

INPUT_PATH, OUTPUT_DIR

(WindowsPath('c:/Users/sezen/Projects/RetailDemo/retail-self-refreshing-report/data/online_retail_II.xlsx'),
 WindowsPath('c:/Users/sezen/Projects/RetailDemo/retail-self-refreshing-report/reports'))

## 3. Read Excel Workbook into a DataFrame

In [3]:
sheets = pd.read_excel(INPUT_PATH, sheet_name=None)
df = pd.concat(
    [sheet.assign(source_sheet=sheet_name) for sheet_name, sheet in sheets.items()],
    ignore_index=True,
)

print(f"Loaded {len(sheets)} sheets and {len(df):,} rows.")
df.head()

Loaded 2 sheets and 1,067,371 rows.


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,source_sheet
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,Year 2009-2010
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,Year 2009-2010
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,Year 2009-2010


## 4. Inspect the DataFrame

In [4]:
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.info()
df.head(10)

Shape: (1067371, 9)
Columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country', 'source_sheet']
<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 9 columns):
 #   Column        Non-Null Count    Dtype         
---  ------        --------------    -----         
 0   Invoice       1067371 non-null  object        
 1   StockCode     1067371 non-null  object        
 2   Description   1062989 non-null  object        
 3   Quantity      1067371 non-null  int64         
 4   InvoiceDate   1067371 non-null  datetime64[us]
 5   Price         1067371 non-null  float64       
 6   Customer ID   824364 non-null   float64       
 7   Country       1067371 non-null  str           
 8   source_sheet  1067371 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(2)
memory usage: 73.3+ MB


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,source_sheet
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,Year 2009-2010
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,Year 2009-2010
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,Year 2009-2010
5,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,13085.0,United Kingdom,Year 2009-2010
6,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,Year 2009-2010
7,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01 07:45:00,5.95,13085.0,United Kingdom,Year 2009-2010
8,489435,22350,CAT BOWL,12,2009-12-01 07:46:00,2.55,13085.0,United Kingdom,Year 2009-2010
9,489435,22349,"DOG BOWL , CHASING BALL DESIGN",12,2009-12-01 07:46:00,3.75,13085.0,United Kingdom,Year 2009-2010


## 5. Validate Expected Columns

In [5]:
expected_columns = {
    "Invoice",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "Price",
    "Customer ID",
    "Country",
    "source_sheet",
}
missing_columns = expected_columns.difference(df.columns)
extra_columns = set(df.columns).difference(expected_columns)

assert not missing_columns, f"Missing columns: {sorted(missing_columns)}"
print("Schema is valid.")
print("Extra columns:", sorted(extra_columns) if extra_columns else "None")

Schema is valid.
Extra columns: None


## 6. Convert Data Types

In [6]:
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["Price"] = pd.to_numeric(df["Price"], errors="coerce")
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
df["Revenue"] = df["Quantity"] * df["Price"]

df[["Quantity", "Price", "InvoiceDate", "Revenue"]].dtypes

Quantity                int64
Price                 float64
InvoiceDate    datetime64[us]
Revenue               float64
dtype: object

In [13]:
def column_extremes(column):
    values = df[column].dropna()
    if values.empty:
        return None, None
    if pd.api.types.is_object_dtype(values) or pd.api.types.is_string_dtype(values):
        values = values.astype("string")
    return values.min(), values.max()

extremes = [column_extremes(column) for column in df.columns]
extreme_values = pd.DataFrame(
    {
        "column": df.columns,
        "minimum": [minimum for minimum, _ in extremes],
        "maximum": [maximum for _, maximum in extremes],
        "non_null_values": [df[column].notna().sum() for column in df.columns],
    }
)

extreme_values

,column,minimum,maximum,non_null_values
0,Invoice,489434,C581569,1067371
1,StockCode,10002,m,1067371
2,Description,DOORMAT UNION JACK GUNS AND ROSES,wrongly sold sets,1062989
3,Quantity,-80995,80995,1067371
4,InvoiceDate,2009-12-01 07:45:00,2011-12-09 12:50:00,1067371
5,Price,-53594.36,38970.0,1067371
6,Customer ID,12346.0,18287.0,824364
7,Country,Australia,West Indies,1067371
8,source_sheet,Year 2009-2010,Year 2010-2011,1067371
9,Revenue,-168469.6,168469.6,1067371


## 7. Identify Invalid Rows

In [7]:
df["is_cancelled"] = df["Invoice"].astype("string").str.upper().str.startswith("C", na=False)

missing_invoice = df["Invoice"].isna()
missing_stock_code = df["StockCode"].isna()
missing_price = df["Price"].isna()
non_numeric_quantity = df["Quantity"].isna()
invalid_date = df["InvoiceDate"].isna()

invalid_mask = (
    missing_invoice
    | missing_stock_code
    | missing_price
    | non_numeric_quantity
    | invalid_date
)
invalid_rows = df.loc[invalid_mask].copy()
invalid_rows["quality_flag"] = "invalid_core_data"
invalid_rows.loc[df.loc[invalid_mask, "Customer ID"].isna(), "quality_flag"] = "missing_customer_id"

print(f"Flagged rows: {len(invalid_rows):,}")
invalid_rows.head()

Flagged rows: 0


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,source_sheet,Revenue,is_cancelled,quality_flag


## 8. Review the First Rows and Missing Values

In [8]:
display(df.head(10))
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_values"))
display(df["Country"].value_counts(dropna=False).head(20).to_frame("row_count"))

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,source_sheet,Revenue,is_cancelled
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,Year 2009-2010,83.4,False
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010,81.0,False
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010,81.0,False
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,Year 2009-2010,100.8,False
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,Year 2009-2010,30.0,False
5,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,13085.0,United Kingdom,Year 2009-2010,39.6,False
6,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,Year 2009-2010,30.0,False
7,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01 07:45:00,5.95,13085.0,United Kingdom,Year 2009-2010,59.5,False
8,489435,22350,CAT BOWL,12,2009-12-01 07:46:00,2.55,13085.0,United Kingdom,Year 2009-2010,30.6,False
9,489435,22349,"DOG BOWL , CHASING BALL DESIGN",12,2009-12-01 07:46:00,3.75,13085.0,United Kingdom,Year 2009-2010,45.0,False


,missing_values
Customer ID,243007
Description,4382
Invoice,0
StockCode,0
Quantity,0
InvoiceDate,0
Price,0
Country,0
source_sheet,0
Revenue,0


,row_count
Country,
United Kingdom,981330
EIRE,17866
Germany,17624
France,14330
Netherlands,5140
Spain,3811
Switzerland,3189
Belgium,3123
Portugal,2620


## 9. Aggregate Sales Metrics

In [10]:
sales_rows = df.loc[~df["is_cancelled"] & df["Revenue"].notna()].copy()
sales_rows["month"] = sales_rows["InvoiceDate"].dt.to_period("M").astype("string")

monthly_revenue = sales_rows.groupby("month", as_index=False)["Revenue"].sum().sort_values("month")
top_products = (
    sales_rows.groupby("Description", dropna=False)["Revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(20)
    .rename("Revenue")
    .reset_index()
)
top_countries = (
    sales_rows.groupby("Country", dropna=False)["Revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(20)
    .rename("Revenue")
    .reset_index()
)

metrics = pd.Series(
    {
        "revenue": sales_rows["Revenue"].sum(),
        "orders": sales_rows["Invoice"].nunique(),
        "customers": sales_rows["Customer ID"].nunique(),
        "cancelled_rows": int(df["is_cancelled"].sum()),
        "flagged_rows": len(invalid_rows),
    },
    name="value",
)
display(metrics.to_frame())
display(monthly_revenue.head())
display(top_products)
display(top_countries)

,value
revenue,2.081392e+07
orders,4.533600e+04
customers,5.881000e+03
cancelled_rows,1.949400e+04
flagged_rows,0.000000e+00


,month,Revenue
0,2009-12,825685.760
1,2010-01,652708.502
2,2010-02,553339.736
3,2010-03,833570.131
4,2010-04,627934.632


,Description,Revenue
0,REGENCY CAKESTAND 3 TIER,344563.25
1,Manual,340731.33
2,DOTCOM POSTAGE,322657.48
3,WHITE HANGING HEART T-LIGHT HOLDER,266923.55
4,"PAPER CRAFT , LITTLE BIRDIE",168469.60
5,JUMBO BAG RED RETROSPOT,150935.56
6,PARTY BUNTING,149187.05
7,ASSORTED COLOUR BIRD ORNAMENT,132187.92
8,POSTAGE,127597.42
9,PAPER CHAIN KIT 50'S CHRISTMAS,123141.54


,Country,Revenue
0,United Kingdom,1.771230e+07
1,EIRE,6.644318e+05
2,Netherlands,5.542323e+05
3,Germany,4.312625e+05
4,France,3.569446e+05
5,Australia,1.699681e+05
6,Spain,1.091785e+05
7,Switzerland,1.010113e+05
8,Sweden,9.190372e+04
9,Denmark,6.986219e+04


## 10. Save Processed Outputs

In [14]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

invalid_rows.to_csv(OUTPUT_DIR / "notebook_invalid_rows.csv", index=False)
monthly_revenue.to_csv(OUTPUT_DIR / "notebook_monthly_revenue.csv", index=False)
top_products.to_csv(OUTPUT_DIR / "notebook_top_products.csv", index=False)
top_countries.to_csv(OUTPUT_DIR / "notebook_top_countries.csv", index=False)
sales_rows.to_csv(OUTPUT_DIR / "notebook_sales_rows.csv", index=False)
extreme_values.to_csv(OUTPUT_DIR / "notebook_extreme_values.csv", index=False)

print(f"Saved notebook outputs to {OUTPUT_DIR.resolve()}")

Saved notebook outputs to C:\Users\sezen\Projects\RetailDemo\retail-self-refreshing-report\reports
